# Ariketa 4.4 — Ez-linealtasuna, overfitting eta erregularizazioa 🌳⚖️ (EGITEKO)

Lehen hiru ariketetan erregresio lineala landu dugu:

```text
ŷ = w₁x₁ + w₂x₂ + ... + b
```

Baina munduko erlazio guztiak ez dira linealak.

Gaur hiru galdera:

```text
1. Zer gertatzen da erlazioa ez bada lineala?
2. Zer gertatzen da modeloa gehiegi konplexua bada?
3. Nola kontrola ditzakegu koefiziente handiak?
```

Landuko dugu:

```text
Linear Regression
      ↓
Decision Tree Regressor
      ↓
Random Forest Regressor
      ↓
OVERFITTING
      ↓
Ridge / Lasso
```

### Nola lan egin

- 🟢 **KOMENTATU:** azaldu zure hitzekin.
- 🟡 **HUTSUNEAK BETE:** `____` osatu.
- 🔴 **KODETU:** zati labur bat idatzi zerotik.

> **Ez dugu hemen hiperparametro-tuning edo Cross-Validation metodologiarik landuko.** Horiek balidazio-unitatean sakonduko ditugu.


# 4.1 — Datuak prestatu 🔎

Exercise 2-ko sei feature berberak erabiliko ditugu:

```text
weight
horsepower
displacement
cylinders
acceleration
model_year
```

eta target-a:

```text
mpg
```

Gogoratu aurreko zuzenketa:

> **Lehenengo erabiltzen ditugun zutabeak hautatu, eta gero `dropna()`.**


In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    r2_score
)


df_raw = sns.load_dataset(
    "mpg"
)

ezaugarriak = [
    "weight",
    "horsepower",
    "displacement",
    "cylinders",
    "acceleration",
    "model_year"
]


# ============================================================
# 🟡 HUTSUNEAK BETE
# ============================================================

df = df_raw[
    ezaugarriak
    + ["____"]
].copy()

df = df.____()


X = df[
    ezaugarriak
]

y = df[
    "mpg"
]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


print(
    "TRAIN:",
    X_train.shape
)

print(
    "TEST:",
    X_test.shape
)


# ============================================================
# 🟢 KOMENTATU
# ============================================================
#
# Zergatik ez dugu df_raw.dropna() zuzenean egiten?
#
# ERANTZUNA:
#


# 4.2 — Lineala vs ez-lineala: zer esan nahi du? 📉🌳

Linear Regression-ek hiperplano lineal bat ikasten du.

Decision Tree batek, aldiz:

```text
if feature <= threshold:
    ...
else:
    ...
```

erabakiak kateatzen ditu.

Erregresio-zuhaitz batean, hosto bakoitzean **zenbakizko balio bat** iragartzen da, normalean hostora iritsi diren TRAIN balioen batezbestekoarekin lotuta.

Lehenengo `weight` bakarrarekin ikusiko dugu geometrikoki:

- Linear Regression → zuzen leuna.
- Decision Tree → zatika konstantea den predikzioa.


In [ ]:
Xw_train = X_train[
    ["weight"]
]

lineal_1d = LinearRegression().fit(
    Xw_train,
    y_train
)

zuhaitz_1d = DecisionTreeRegressor(
    max_depth=3,
    random_state=42
).fit(
    Xw_train,
    y_train
)


x_line = np.linspace(
    X_train["weight"].min(),
    X_train["weight"].max(),
    300
)

X_line = pd.DataFrame({
    "weight": x_line
})


plt.figure(
    figsize=(9, 5)
)

plt.scatter(
    X_train["weight"],
    y_train,
    alpha=0.25,
    label="TRAIN datuak"
)

plt.plot(
    x_line,
    lineal_1d.predict(
        X_line
    ),
    linewidth=2.5,
    label="Linear Regression"
)

plt.plot(
    x_line,
    zuhaitz_1d.predict(
        X_line
    ),
    linewidth=2.5,
    label="Decision Tree (depth=3)"
)

plt.xlabel(
    "Pisua (lb)"
)

plt.ylabel(
    "mpg"
)

plt.title(
    "Lineala vs zuhaitzaren predikzio zatitua"
)

plt.legend()
plt.grid(
    alpha=0.2
)

plt.show()


# 4.3 — Hiru eredu, problema bera 📊

Orain sei feature-ak erabiliko ditugu.

Konparatuko ditugu:

```text
Linear Regression
Decision Tree (max_depth=3)
Random Forest
```

Ez gara “irabazle unibertsal” bat bilatzen.

Galdera:

> **Split honetan eta konfigurazio hauetan, zer portaera dute?**

Erregresio-metrikak gogoratu:

```text
RMSE txikiagoa → hobea
R² handiagoa   → hobea
```


In [ ]:
lineala = LinearRegression()

zuhaitza = DecisionTreeRegressor(
    max_depth=3,
    random_state=42
)


# ============================================================
# 🔴 KODETU — RANDOM FOREST
# ============================================================
#
# Sortu RandomForestRegressor:
#
# n_estimators = 200
# random_state = 42
# n_jobs = -1
#
# eta gorde `basoa` aldagaian.


# IDATZI HEMEN:






# ============================================================
# MODELOAK ENTRENATU
# ============================================================

lineala.fit(
    X_train,
    y_train
)

zuhaitza.fit(
    X_train,
    y_train
)

basoa.fit(
    X_train,
    y_train
)


def rmse(
    y_true,
    y_pred
):
    return np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )


lerroak = []

for izena, modeloa in [
    ("Linear Regression", lineala),
    ("Decision Tree d=3", zuhaitza),
    ("Random Forest", basoa)
]:

    pred_test = modeloa.predict(
        X_test
    )

    lerroak.append({
        "Modeloa": izena,
        "TEST RMSE": rmse(
            y_test,
            pred_test
        ),
        "TEST R²": r2_score(
            y_test,
            pred_test
        )
    })


emaitzak = pd.DataFrame(
    lerroak
)

display(
    emaitzak.round(3)
)


# ============================================================
# 🟢 KOMENTATU
# ============================================================
#
# Esperimentu honetan, zein modelok lortu du TEST R² handiena?
#
# ERANTZUNA:
#
#
# Esan al dezakegu horregatik Random Forest beti hobea dela?
#
# ERANTZUNA:
#


# 4.4 — Random Forest Regressor barrutik 🌲

Sailkapenean ikusi genuen bezala, forest-ak zuhaitz asko ditu.

Baina erregresioan azken emaitza ez da klase-bozketa:

> **zuhaitzen zenbakizko iragarpenak batezbestekotu egiten dira.**

Hori zuzenean egiaztatuko dugu TEST-eko errenkada batekin.


In [ ]:
x0 = X_test.iloc[
    [0]
]


# Zuhaitz GUZTIEN iragarpenak
tree_preds = np.array([
    estimator.predict(
        x0
    )[0]
    for estimator in basoa.estimators_
])


forest_pred = basoa.predict(
    x0
)[0]


print(
    "Lehen 10 zuhaitzen iragarpenak:"
)

print(
    np.round(
        tree_preds[:10],
        2
    )
)


print(
    "\nZuhaitz guztien batezbestekoa:",
    round(
        tree_preds.mean(),
        3
    )
)

print(
    "RandomForestRegressor.predict():",
    round(
        forest_pred,
        3
    )
)


# ============================================================
# 🟢 KOMENTATU
# ============================================================
#
# Zergatik dira bi azken zenbakiak ia berdinak?
#
# ERANTZUNA:
#


# 4.5 — `feature_importances_`: zer esaten du eta zer EZ? 🔎

Random Forest-ek feature bakoitzaren impurity-reduction ekarpena agregatzen du.

Interpretazio egokia:

> **Forest honetan, split erabilgarriak sortzeko zein feature izan dira garrantzitsuenak?**

Interpretazio desegokia:

> ❌ “importance = 0.30 → mpg-ren %30 eragiten du”

Eta:

> ❌ feature importance **ez da kausalitatea**.


In [ ]:
importances = pd.Series(
    basoa.feature_importances_,
    index=ezaugarriak
).sort_values(
    ascending=False
)


display(
    importances.round(3)
)


importances.sort_values().plot(
    kind="barh",
    figsize=(8, 4)
)

plt.xlabel(
    "Impurity-based feature importance"
)

plt.title(
    "Random Forest Regressor — feature_importances_"
)

plt.tight_layout()
plt.show()


# ============================================================
# 🟢 KOMENTATU
# ============================================================
#
# 1. Zein dira lehen 3 feature-ak forest honen arabera?
#
# ERANTZUNA:
#
#
# 2. Feature batek importance=0.30 badu, mpg-ren %30 "eragiten" du?
#
# ERANTZUNA:
#


# 4.6 — Overfitting: TRAIN perfektua ≠ modelo ona ⚠️

Bi zuhaitz aurrez definituko ditugu:

```text
Zuhaitz sinplea → max_depth=3
Zuhaitz sakona  → max_depth=None
```

Ez dugu `max_depth` optimizatuko.

Helburua soilik hau ikustea da:

> **konplexutasun handiak TRAIN memorizatzea ekar dezake, baina TEST generalizazioa okerragoa izan daiteke.**


In [ ]:
zuhaitz_sakona = DecisionTreeRegressor(
    random_state=42
)

zuhaitz_sakona.fit(
    X_train,
    y_train
)


modelo_overfit = [
    ("Zuhaitz d=3", zuhaitza),
    ("Zuhaitz sakona", zuhaitz_sakona)
]


lerroak_overfit = []

for izena, modeloa in modelo_overfit:

    pred_train = modeloa.predict(
        X_train
    )

    pred_test = modeloa.predict(
        X_test
    )

    lerroak_overfit.append({
        "Modeloa": izena,
        "TRAIN R²": r2_score(
            y_train,
            pred_train
        ),
        "TEST R²": r2_score(
            y_test,
            pred_test
        ),
        "TEST RMSE": rmse(
            y_test,
            pred_test
        )
    })


overfit_df = pd.DataFrame(
    lerroak_overfit
)

display(
    overfit_df.round(3)
)


# ============================================================
# GRAFIKOA
# ============================================================

overfit_df.set_index(
    "Modeloa"
)[
    ["TRAIN R²", "TEST R²"]
].plot(
    kind="bar",
    figsize=(8, 4),
    rot=0
)

plt.ylim(
    0,
    1.05
)

plt.ylabel(
    "R²"
)

plt.title(
    "Overfitting: TRAIN vs TEST"
)

plt.tight_layout()
plt.show()


# ============================================================
# 🟢 KOMENTATU
# ============================================================
#
# Zuhaitz sakonak TRAIN R² ≈ 1 lortzen badu, zergatik ez da
# automatikoki modelo hobea?
#
# ERANTZUNA:
#


# 4.7 — Ridge eta Lasso: koefiziente handiak zigortu ⚖️

Orain berriro **eredu linealetara** itzuliko gara.

Linear Regression-ek MSE minimizatzen du.

Ridge-k gehitzen du:

$$
MSE + \alpha \sum_j w_j^2
$$

Lasso-k gehitzen du:

$$
MSE + \alpha \sum_j |w_j|
$$

Intuizioa:

```text
Ridge (L2)
→ koefiziente handiak txikitu

Lasso (L1)
→ koefizienteak txikitu
→ batzuk 0 bihur daitezke
```

### Scaling zergatik?

Feature-ak unitate oso desberdinetan daude:

```text
weight → milaka lb
cylinders → 4, 6, 8...
```

Penalty bera modu bidezkoan aplikatzeko, Ridge/Lasso aurretik feature-ak eskalatzea **oso gomendagarria** da.

`StandardScaler.fit()` → TRAIN bakarrik.


In [ ]:
scaler = StandardScaler()

X_train_s = scaler.fit_transform(
    X_train
)

X_test_s = scaler.transform(
    X_test
)


lineal_s = LinearRegression().fit(
    X_train_s,
    y_train
)


# ============================================================
# 🟡 HUTSUNEAK BETE
# ============================================================

ridge = ____(
    alpha=10
).fit(
    X_train_s,
    y_train
)


lasso = ____(
    alpha=1.0,
    max_iter=10000
).fit(
    X_train_s,
    y_train
)


coef_df = pd.DataFrame({
    "Linear": lineal_s.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_
}, index=ezaugarriak)


display(
    coef_df.round(3)
)


coef_df.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.axhline(
    0,
    linewidth=1
)

plt.ylabel(
    "Koefizientea (scaled features)"
)

plt.title(
    "Linear vs Ridge vs Lasso — koefizienteak"
)

plt.tight_layout()
plt.show()


print(
    "Lasso-k zerora eramandako koefizienteak:",
    int(
        np.sum(
            np.isclose(
                lasso.coef_,
                0
            )
        )
    )
)


# ============================================================
# 🟢 KOMENTATU
# ============================================================
#
# 1. Zer egiten du Ridge-k koefizienteekin?
#
# ERANTZUNA:
#
#
# 2. Zer berezitasun du Lasso-k?
#
# ERANTZUNA:
#


# 4.8 — Erregularizazioak score-a beti hobetzen du? 🤔

Ez.

Erregularizazioaren helburua:

> **modeloa sinplifikatzea / koefizienteak kontrolatzea / overfitting-arriskua murriztea**

Ez da:

> ❌ “TEST score-a beti handitzea”.

`alpha=10` eta `alpha=1` balioak **adibide didaktikoak** dira; ez ditugu hemen tuning bidez aukeratuko.


In [ ]:
lerroak_reg = []

for izena, modeloa in [
    ("Linear", lineal_s),
    ("Ridge α=10", ridge),
    ("Lasso α=1", lasso)
]:

    pred = modeloa.predict(
        X_test_s
    )

    lerroak_reg.append({
        "Modeloa": izena,
        "TEST RMSE": rmse(
            y_test,
            pred
        ),
        "TEST R²": r2_score(
            y_test,
            pred
        )
    })


reg_df = pd.DataFrame(
    lerroak_reg
)

display(
    reg_df.round(3)
)


# ============================================================
# 🟢 KOMENTATU
# ============================================================
#
# Regularizatutako modelo batek TEST R² baxuagoa lortzen badu
# esperimentu honetan, esan nahi al du erregularizazioa "txarra" dela?
#
# ERANTZUNA:
#


# 4.9 — Deep Learning-erako beste zubi bat 🧠

Ridge-ren ideia:

```text
loss
+
pisu handien zigorra
```

Sare neuronaletan ere antzeko ideia erabiliko dugu:

```text
L2 regularization / weight decay
```

Ez dira testuinguru guztietan formula bera, baina intuizio bera dago:

> **pisuak gehiegi haztea saihestu eta modeloaren konplexutasuna kontrolatu.**

### 🟢 Osatu mental model-a

```text
Decision Tree
→ feature-space-a __________
→ hostoan zenbakizko iragarpena

Random Forest Regressor
→ zuhaitz asko
→ haien iragarpenen __________

Ridge
→ __________ penalty

Lasso
→ __________ penalty
→ koefiziente batzuk 0 izan daitezke
```


# Ondorioak ✅

Ariketa honetan lau ideia nagusi lotu ditugu:

```text
LINEARITATEA
Linear Regression
        ↓
EZ-LINEALTASUNA
Decision Tree / Random Forest
        ↓
KONPLEXUTASUN GEHIEGI
Overfitting
        ↓
KONPLEXUTASUNA KONTROLATU
Ridge / Lasso
```

Gakoak:

1. **Decision Tree Regressor**-ek feature-space-a zatitzen du eta zenbakizko balioak iragartzen ditu.
2. **Random Forest Regressor**-ek zuhaitz askoren iragarpenak batezbestekotzen ditu.
3. `feature_importances_` erabilgarria da, baina **ez da kausalitatea**.
4. **TRAIN perfektua ez da helburua**; generalizazioa da helburua.
5. **Ridge (L2)** koefizienteak txikitzera bultzatzen ditu.
6. **Lasso (L1)** koefiziente batzuk 0 bihur ditzake.
7. Ridge/Lasso erabiltzean scaling-a oso gomendagarria da penalty-a feature guztiei modu bidezkoan aplikatzeko.
8. Erregularizazioak **ez du score hobea automatikoki bermatzen**.
9. Ez dugu TEST erabili hiperparametroak aukeratzeko; hori geroagoko balidazio-unitatearen lana izango da.
10. L2 erregularizazioaren intuizioa berriro agertuko da **Deep Learning**-en.

Hurrengo eta azken UD4 ariketan, orain arteko ideiak bildu eta **sare neuronaletarako azken zubia** egingo dugu.
